In [ ]:
    ############    #############   Pydantic and Dependency Injection   #############   ##############   

 =>  Pydantic models validate untrusted input (request bodies, config, tool-call arguments)
       at the boundary, and raise a clear error instead of letting bad data drift deeper into
       the system.

 =>  Dependency injection means a function/class receives its collaborators (a DB session, an
       HTTP client, a settings object) from the outside instead of constructing them itself --
       this is what makes services swappable and testable (inject a fake in tests).


In [ ]:
from pydantic import BaseModel, field_validator

class CreateUserRequest(BaseModel):
    email: str
    age: int

    @field_validator("email")
    @classmethod
    def email_must_look_valid(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError("not a valid email")
        return v

    @field_validator("age")
    @classmethod
    def must_be_adult(cls, v: int) -> int:
        if v < 18:
            raise ValueError("user must be 18+")
        return v

# valid
print(CreateUserRequest(email="a@b.com", age=25))

# invalid -- raises pydantic.ValidationError with a precise, field-level error message
try:
    CreateUserRequest(email="not-an-email", age=12)
except Exception as exc:
    print("validation failed as expected:\n", exc)


In [ ]:
 =>  This uses a plain str + custom field_validator for the email check to avoid an extra
       dependency -- in a real FastAPI app, prefer 'email: EmailStr' from pydantic, which
       does full RFC-compliant validation (requires the 'email-validator' package installed
       alongside pydantic).


In [ ]:
class Notifier:
    def send(self, msg: str) -> None:
        print(f"[real notifier] {msg}")

class FakeNotifier:
    def __init__(self):
        self.sent = []

    def send(self, msg: str) -> None:
        self.sent.append(msg)

class SignupService:
    def __init__(self, notifier: Notifier):  # injected, not constructed internally
        self._notifier = notifier

    def signup(self, email: str) -> None:
        self._notifier.send(f"welcome {email}")

# production: SignupService(Notifier())
# test:       SignupService(FakeNotifier())  <- no real emails sent
fake = FakeNotifier()
SignupService(fake).signup("a@b.com")
print(fake.sent)


In [ ]:
 =>  FastAPI's 'Depends()' is this exact same pattern wired into the framework: a route
       declares what it needs (a DB session, the current user, settings) and FastAPI supplies
       it, so routes stay thin and testable.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Install pydantic's email-validator extra and swap the manual email check for a
           real 'EmailStr' field.

 =>  [ ] Build a small FastAPI route that uses 'Depends()' to inject a repository, and write
           a test that injects a fake repository instead of a real database.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Constructing dependencies (a DB connection, an HTTP client) directly inside a class's
       __init__ -- this makes the class impossible to unit test without hitting the real
       dependency.

 =>  Validating input in application code that duplicates what pydantic could enforce at the
       boundary -- keep validation in one place (the model), not scattered through the code.
